In [ ]:
import os, glob
import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneOut
import xgboost as xgb
from sklearn.metrics import recall_score, precision_score, balanced_accuracy_score, roc_auc_score

In [25]:
DATA_PATH   = "/home/justine/code/Maelle05/DyslexIA/Dataset"
LABELS_PATH = "/home/justine/code/Maelle05/DyslexIA/Dataset/dyslexia_class_label.csv"

GAZE_COLS  = ['x_left', 'y_left', 'x_right', 'y_right']

files_raw = glob.glob(os.path.join(DATA_PATH, "*clean.csv"))
labels = pd.read_csv(LABELS_PATH)

In [26]:
def to_uniform(signal, timestamps):
    t_norm = (timestamps - timestamps[0]) / (timestamps[-1] - timestamps[0])
    x_new  = np.linspace(0, 1, len(signal))
    return np.interp(x_new, t_norm, signal)

def extract_features(df, gaze_cols):
    timestamps = df['t_s'].values.astype(float)
    signals = {}
    for col in gaze_cols:
        signals[col] = to_uniform(df[col].values.astype(float), timestamps)

    # Divergence binoculaire moyenne
    mean_cross_divergence_x = np.mean(np.abs(signals['x_left'] - signals['x_right']))

    # Cyclope
    x = (signals['x_left'] + signals['x_right']) / 2
    y = (signals['y_left'] + signals['y_right']) / 2

    # Vélocités
    vel_x = np.abs(np.diff(x))
    x_vel_p90p50 = np.percentile(vel_x, 90) / (np.percentile(vel_x, 50) + 1e-8)

    vel_y = np.abs(np.diff(y))
    y_vel_p90p50 = np.percentile(vel_y, 90) / (np.percentile(vel_y, 50) + 1e-8)

    # Taux de changements de direction
    dx = np.diff(x)
    x_direction_changes = np.sum(np.diff(np.sign(dx)) != 0) / len(dx)

    return np.array([mean_cross_divergence_x,
                     x_vel_p90p50, y_vel_p90p50,
                     x_direction_changes])

In [27]:
rows = []
for f in files_raw:
    df = pd.read_csv(f)
    sid = os.path.basename(f).split('_')[1]
    label_row = labels[labels['subject_id'] == sid]
    rows.append({'sid': sid,
                 'label': label_row['class_id'].values[0],
                 'features': extract_features(df, GAZE_COLS)})

sids = np.array([r['sid']      for r in rows])
X    = np.array([r['features'] for r in rows])
y    = np.array([r['label']    for r in rows])
print(f"X : {X.shape} | Classes : {np.bincount(y)} (0=non-dys, 1=dys)")

X : (255, 4) | Classes : [123 132] (0=non-dys, 1=dys)
